# সহজ ভাষায় Notebook Guide

এই notebook project-এর **একই `src/` module** (API, CLI আর UI যেগুলো ব্যবহার করে) cell-by-cell চালিয়ে Offline Market Intelligence Team খুলে দেখায়: snapshot থেকে দাম আর news, code-এ risk metric, FinBERT sentiment, 0-100 Risk Score, দুই agent-এর crew, code-এর verification আর PDF memo।

## কীভাবে ব্যবহার করবেন?

1. `notebooks/` folder থেকে উপর থেকে নিচে run করুন।
2. Section 1-6 আর 9 শুধু Python আর SQLite — Ollama বা internet লাগে না।
3. Section 4-এর FinBERT cell প্রথমবার ~440 MB model download করে। `transformers` না থাকলে একটা keyword stand-in দিয়ে চলে — তখন sentiment-এর সংখ্যা শুধু উদাহরণ।
4. Section 7-8-এ live `llama3.1:8b` (Ollama) লাগে। এই notebook যে machine-এ লেখা, সেখানে Ollama ছিল না, তাই কোনো output save করা নেই (ADR 0008) — নিজের মেশিনে চালিয়ে দেখুন।

> Educational project, synthetic data, not investment advice.

# Offline Market Intelligence Team — Exploration

1. Offline Snapshot — synthetic দাম আর news
2. SQLite store — local data persistence
3. Risk metric হাতে-কলমে — volatility, drawdown, VaR
4. FinBERT sentiment আর cache
5. Risk Score — normalize, weight, band
6. Verifier — agent-এর ভুল সংখ্যা আর বানানো id ধরা
7. Live: grounded crew (News Scout Agent → Risk Analyst Agent)
8. Live: tool mode — agent নিজে tool ডাকলে কী হয়
9. PDF memo — LLM ছাড়া

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import config  # sets CREWAI_DISABLE_TELEMETRY etc. BEFORE crewai is imported
import json
import logging
import math
import sqlite3
from dataclasses import replace

import pandas as pd

from config import get_config
from src.crew.team import facts_text
from src.flow.market_flow import run_analysis
from src.llm.factory import ollama_status
from src.memo.pdf import write_memo_pdf
from src.risk.metrics import annualized_vol, assess, compute_risk, historical_var, max_drawdown
from src.sentiment.finbert import FinBert, SentimentUnavailable, score_news
from src.store.db import MarketStore
from src.verify.memo_check import verify_memo

logging.getLogger("crewai").setLevel(logging.ERROR)
pd.set_option("display.max_colwidth", 90)
cfg = get_config()
store = MarketStore(cfg.data.db_path, cfg.data.snapshot_dir)   # first run loads data/snapshot automatically
print(ollama_status())

## 1. Offline Snapshot

`data/make_snapshot.py` একটা seed দিয়ে ৮টা **fictional** company-র এক বছরের (২৫২ trading day) daily close বানায় — geometric Brownian motion, প্রতিটা ticker-এর নিজের drift আর volatility। HELX-এ এক দিনে −৩৮% crash (trial failure), VALT-এ আট দিনের slide (regulatory probe)। প্রতিটা ticker-এর ৬-৭টা বানানো headline, id `news:ACME:3`-এর মতো stable। একই seed দিলে byte-by-byte একই file আসে — test এটা যাচাই করে।

In [ ]:
manifest = json.loads((cfg.data.snapshot_dir / "manifest.json").read_text(encoding="utf-8"))
print(manifest["disclaimer"])
pd.DataFrame(manifest["tickers"])[["ticker", "name", "sector", "drift", "volatility", "shock_days"]]

In [ ]:
pd.DataFrame(store.news("HELX"))[["id", "date", "headline"]]

## 2. SQLite Store — Local Data Persistence

Server ছাড়া একটা file-ই database (`data/market.db`, git-ignored)। পাঁচটা table: `tickers`, `prices`, `news`, `sentiment` (FinBERT cache), `memos` (PDF index)। Optional `refresh` (yfinance + RSS) real ticker এই একই table-এ লেখে; বাকি pipeline একটুও বদলায় না।

In [ ]:
con = sqlite3.connect(store.db_path)
print([r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type = 'table'")])
print(con.execute("SELECT ticker, COUNT(*), MIN(close), MAX(close) FROM prices GROUP BY ticker").fetchall())
con.close()
pd.DataFrame(store.tickers())

## 3. Risk Metric হাতে-কলমে

তিনটা metric, তিনটাই code-এ:

* **Annualized volatility** = daily log return-এর sample stdev × √252
* **Max drawdown** = running peak থেকে সবচেয়ে বড় পতন, `1 - price / peak`
* **95% historical VaR (1-day)** = daily simple return-এর 5th percentile, loss হিসেবে positive

নিচে HELX-এর volatility প্রথমে সূত্র ধরে হাতে, তারপর project-এর function দিয়ে — দুটো মিলতে হবে।

In [ ]:
prices = store.closes("HELX")
closes = [c for _, c in prices]
lr = [math.log(b / a) for a, b in zip(closes, closes[1:])]
mean = sum(lr) / len(lr)
vol_by_hand = math.sqrt(sum((x - mean) ** 2 for x in lr) / (len(lr) - 1)) * math.sqrt(252)
worst = min(range(1, len(closes)), key=lambda i: closes[i] / closes[i - 1])
print(f"volatility by hand {vol_by_hand:.4f} | function {annualized_vol(closes):.4f}")
print(f"max drawdown {max_drawdown(closes):.1%} | 95% 1-day VaR {historical_var(closes):.2%}")
print(f"worst day {prices[worst][0]}: {closes[worst] / closes[worst - 1] - 1:+.1%}")

In [ ]:
rows = []
for t in manifest["tickers"]:
    c = [x for _, x in store.closes(t["ticker"])]
    rows.append({"ticker": t["ticker"], "generator sigma": t["volatility"], "measured vol": round(annualized_vol(c), 3),
                 "max drawdown": round(max_drawdown(c), 3), "VaR 95%": round(historical_var(c), 4)})
pd.DataFrame(rows)

Crash নেই এমন ticker-এ measured volatility generator-এর sigma-র কাছাকাছি। HELX-এর একটা crash দিন একাই volatility, drawdown আর VaR তিনটাই বাড়িয়ে দেয় — historical metric অতীতের ব্যথা মনে রাখে, কিন্তু ভবিষ্যতের crash আগে থেকে দেখে না।

## 4. FinBERT Sentiment আর Cache

Headline-এর sentiment দেয় `ProsusAI/finbert` — financial text-এ trained, natively positive / negative / neutral (ADR 0009)। Model প্রথম **cache miss**-এ load হয়; প্রতিটা headline-এর ফল SQLite-এ cache হয়, তাই দ্বিতীয়বার model লাগে না।

| Model | Use Case | Advantage | Limitation |
|---|---|---|---|
| **`ProsusAI/finbert`** (chosen) | Financial headline sentiment | Finance domain, native 3-class, probability দেয় | ~440 MB, শুধু English |
| `cardiffnlp/twitter-roberta-base-sentiment-latest` | Tweet / review | 3-class, Week 10-এ দেখা | Tweet domain — finance phrasing-এ domain shift |
| `mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis` | হালকা financial sentiment | ছোট, দ্রুত | Community model; calibration নিজে যাচাই করতে হয় |
| `llama3.1:8b` zero-shot | Label-ছাড়া নতুন domain | Extra model লাগে না | ধীর, non-deterministic, calibrated probability নেই |

`transformers` না থাকলে (বা offline প্রথম run-এ) নিচের cell একটা ছোট keyword stand-in নেয় — তখনকার সংখ্যা শুধু pipeline দেখানোর জন্য, আসল sentiment না।

In [ ]:
class KeywordSentiment:
    """Illustrative stand-in when FinBERT is unavailable. NOT a real sentiment model."""
    name = "keyword-standin"
    NEG = ("recall", "misses", "fall", "slide", "investigation", "fined", "cuts", "loss", "downgrade",
           "suspends", "delays", "warns", "reduction", "short seller")
    POS = ("beats", "raises", "wins", "record", "approves", "rally", "jumps", "unveils", "launches",
           "renews", "expands", "settlement", "buyback")

    def __call__(self, texts):
        out = []
        for t in map(str.lower, texts):
            if any(w in t for w in self.NEG):
                out.append({"positive": 0.05, "negative": 0.85, "neutral": 0.10})
            elif any(w in t for w in self.POS):
                out.append({"positive": 0.80, "negative": 0.05, "neutral": 0.15})
            else:
                out.append({"positive": 0.10, "negative": 0.10, "neutral": 0.80})
        return out


try:
    classifier = FinBert()
    print(classifier(["Acme Robotics beats third-quarter earnings estimates"]))
except SentimentUnavailable as e:
    print("FinBERT unavailable ->", e, "\nusing the keyword stand-in (illustrative numbers)")
    classifier = KeywordSentiment()

scored = score_news(store, store.news("HELX"), classifier)
pd.DataFrame(scored)[["id", "date", "label", "positive", "negative", "headline"]]

**নিজে দেখুন:** "explores strategic alternatives, including a sale" — finance-এ এটা প্রায়ই খারাপ খবর, কিন্তু শব্দগুলো নিরপেক্ষ। FinBERT একে কী বলে? Domain model-এরও সীমা আছে।

## 5. Risk Score — Normalize, Weight, Band

প্রতিটা metric config-এর bound দিয়ে 0-100 risk point-এ যায় (linear, বাইরে clamp), তারপর weight দিয়ে যোগ:

| Metric | 0 points | 100 points | Weight |
|---|---|---|---|
| Annualized volatility | 10% | 70% | 0.35 |
| Max drawdown | 5% | 60% | 0.25 |
| 95% 1-day VaR | 1% | 6% | 0.25 |
| Net news sentiment | +1 | −1 | 0.15 |

Band: < 35 low, < 65 medium, বাকি high। Net sentiment-এ শেষ দামের তারিখের (`as_of`) পরের news বাদ যায় — নাহলে look-ahead bias।

In [ ]:
rows = []
for t in store.tickers():
    r, _ = assess(store, t["ticker"], classifier)
    rows.append({"ticker": r.ticker, "score": r.score, "band": r.band, **{f"{k} pts": v for k, v in r.components.items()}})
pd.DataFrame(rows).sort_values("score", ascending=False)

In [ ]:
news_heavy = replace(cfg.risk, weights={"volatility": 0.25, "max_drawdown": 0.20, "var_95": 0.15, "sentiment": 0.40})
for t in ("ACME", "VALT", "HELX"):
    r, news = assess(store, t, classifier)
    alt = compute_risk(t, store.closes(t), news, news_heavy)
    print(f"{t}: default {r.score:5.1f} ({r.band:6}) -> news-heavy {alt.score:5.1f} ({alt.band})")

Weight একটা **policy**, data না। কোন weight কেন, সেটা README-তে লেখা থাকে; agent এটা দেখতে পায়, বদলাতে পারে না।

## 6. Verifier — Agent-এর ভুল সংখ্যা আর বানানো id ধরা

Agent-এর লেখা publish হওয়ার আগে code তিনটা জিনিস দেখে: cited news id আসল কিনা, memo-র Risk Score / volatility / drawdown / VaR computed মানের সাথে rounding-এর মধ্যে মেলে কিনা, আর required section আছে কিনা। নিচের memo-তে Risk Score বানানো, একটা id-ও বানানো।

In [ ]:
risk_obj, news = assess(store, "ACME", classifier)
risk = risk_obj.to_dict()
ids = {n["id"] for n in news}
print(facts_text(risk), "\n")
brief = f"- Outlook raised [{news[-1]['id']}]\n- Gripper sensor recall earlier in the year [{news[3]['id']}]"
bad_memo = ("## Risk Summary\nThe Risk Score is 12.0 out of 100.\n\n## Key Drivers\n"
            f"Annualized volatility of {risk['metrics']['volatility'] * 100:.1f}% is moderate.\n\n"
            "## News Impact\nA rumoured merger helps [news:ACME:42].")
check = verify_memo(brief, bad_memo, risk, ids)
print(check.summary())
print("\nfeedback for the next round:\n" + check.feedback())

In [ ]:
good_memo = bad_memo.replace("12.0", f"{risk['score']:.1f}").replace("[news:ACME:42]", f"[{news[-1]['id']}]")
print(verify_memo(brief, good_memo, risk, ids).summary())

## 7. Live — Grounded Crew (default)

Flow নিজে data, sentiment আর Risk Score code-এ বানায়, সংখ্যাগুলো Risk Analyst Agent-এর task-এ বসায়। News Scout Agent headline বাছে, Risk Analyst Agent memo লেখে, code verify করে, দরকারে একবার retry, তারপর PDF অথবা needs_human।

> **Output illustrative:** এই cell-এ Ollama + `llama3.1:8b` লাগে (১-৩ মিনিট)। Ollama না থাকলে cell শুধু "Skipped" লেখে। চালানোর আগে লিখে ফেলুন: publish হবে নাকি needs_human, আর কেন।

In [ ]:
ok, msg = ollama_status()
if ok:
    state = run_analysis("ACME", "grounded", store=store, classifier=classifier,
                         on_task=lambda o: print(f"  done: {o.agent} ({len(o.raw)} chars)"))
    print("outcome:", state.outcome, "| rounds:", state.rounds, "|", state.seconds, "s")
    print(state.verification)
    print("\n" + state.brief + "\n\n" + state.memo)
    print("pdf:", state.pdf_path)
else:
    print("Skipped (illustrative section):", msg)

## 8. Live — Tool Mode (failure demo)

এবার Risk Analyst Agent-এর task-এ কোনো সংখ্যা নেই; তাকে `get_risk_metrics` আর `get_news_sentiment` tool দেওয়া। Week 14-এ llama3.1:8b CrewAI-র ভেতরে tool প্রায় ডাকেনি, নিজে result বানিয়েছে। সংখ্যা বানালে verifier `wrong_numbers`-এ ধরে, আর outcome হয় needs_human।

> **Output illustrative:** Ollama লাগে; ফল run থেকে run-এ বদলায়।

In [ ]:
if ok:
    state = run_analysis("ACME", "tool", store=store, classifier=classifier)
    print("outcome:", state.outcome, "| rounds:", state.rounds)
    print("wrong numbers:", state.verification.get("wrong_numbers"), "| error:", state.verification.get("error"))
    print("\n" + state.memo[:1200])
else:
    print("Skipped (illustrative section):", msg)

## 9. PDF Memo — LLM ছাড়া

PDF-এর table আর Risk Score সরাসরি code-এর `risk` dict থেকে আসে; agent-এর লেখা শুধু ব্যাখ্যা। নিচে Section 6-এর সঠিক memo দিয়ে একটা PDF, Ollama ছাড়াই (`output/`, git-ignored)।

In [ ]:
pdf_path = write_memo_pdf(cfg.data.output_dir / "notebook-demo-ACME.pdf", company=store.ticker("ACME")["name"],
                          source="synthetic", risk=risk, brief=brief, memo=good_memo,
                          prices=store.closes("ACME"), mode="notebook (hand-written memo)")
print(pdf_path, pdf_path.stat().st_size, "bytes")

**কী শিখলাম:**

* Risk Score পুরোটা code-এ — একই data দিলে একই সংখ্যা, test করা যায়, আর agent সেটা বদলাতে পারে না।
* Agent-এর কাজ বাছাই আর ব্যাখ্যা। সংখ্যা কপি করতে ভুল হলে verifier ধরে, feedback দিয়ে একবার retry, তারপর মানুষের কাছে।
* ছোট local model-এ tool calling নির্ভরযোগ্য না — তাই default grounded; tool mode শুধু failure দেখাতে।
* Synthetic data দিয়ে শেখা। Real market-এ এই score investment advice না।

## 🧠 Brain Teasers

1. `config.py`-তে volatility-র bound `(0.10, 0.70)` থেকে `(0.10, 0.40)` করুন। কোন ticker-এর band বদলায়? Clamp কোথায় হচ্ছে?
2. `half_life_days` 30 থেকে 7 করুন। কোন ticker-এর sentiment component সবচেয়ে বেশি নড়ে, আর কেন?
3. Section 6-এর memo-তে লিখুন "volatility contributes 35% of the score"। Verifier কী বলে? এটা false alarm কেন, আর কীভাবে ঠিক করবেন?
4. `data/make_snapshot.py`-তে HELX-এর crash (`{180: -0.38}`) সরিয়ে script চালান। Drawdown আর VaR কতটা বদলায়? (পরে `git checkout -- data/snapshot` দিয়ে ফেরত আনুন।)
5. FinBERT আর keyword stand-in কোন headline-এ একমত না? কোনটা ঠিক মনে হয়?